*Owner: **[Data Understanding owner]** — extracted from `airbnb_pricing_crispdm_v7.ipynb` for individual CRISP-DM phase attribution. Run from the repo root (same folder as `db.py`, `pricing_model.py`, etc.).*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from real_data_loader import load


## 2. Data Understanding

Source: Maven Analytics "Airbnb Listings & Reviews" (`listings.csv` and `reviews.csv`), spanning 10
cities across 8 countries and 8 currencies.

**Key Columns Explored**

| Column | Meaning |
|---|---|
| `city` | Market the listing belongs to |
| `price` | Nightly price in the listing's local currency |
| `room_type` | Entire place / Private room / Hotel room / Shared room |
| `bedrooms` | Number of bedrooms |
| `accommodates` | Guest capacity |
| `district` | Sub-city location |
| `host_response_time` / `host_response_rate` | Host responsiveness metrics |
| `host_acceptance_rate` | Share of booking requests the host accepts |
| `host_is_superhost` | Whether the host holds Superhost status |
| `host_since` | Date the host account was created |
| `review_scores_rating` + sub-scores | Guest review ratings across categories |
| `listing_id` (in `reviews.csv`) | Foreign key used to count reviews per listing |


In [ ]:
# Load the two raw Maven Analytics files used throughout this notebook
import pandas as pd
import numpy as np

listings = pd.read_csv("listings.csv", encoding="ISO-8859-1", low_memory=False)
reviews = pd.read_csv("reviews.csv", usecols=["listing_id"])

# Confirm the load worked, and check how listings are spread across the 10 markets
print(f"listings: {listings.shape[0]:,} rows, {listings.shape[1]} columns")
print(f"reviews:  {reviews.shape[0]:,} rows")
listings["city"].value_counts()

**Interpretation:** all 10 expected markets are present, with no missing cities. Listing counts
are heavily imbalanced.

In [ ]:
# dtypes, and a numerical vs categorical breakdown
listings.info()

numeric_cols = listings.select_dtypes(include="number").columns.tolist()
categorical_cols = listings.select_dtypes(include="object").columns.tolist()
print(f"\nNumerical ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

listings[numeric_cols].describe().round(2)

**Interpretation:** `listings.info()` confirms 33 columns split into 19 numerical columns (13 `float64`, 6 `int64`) and 14 categorical (`object`) columns. Notably, `host_since` is stored as an object rather than a parsed date: flagged here as an open item to address in Data Preparation.

In [ ]:
# Uniqueness: Check for duplicate rows and duplicate listing_id values
n_dupe_rows = listings.duplicated().sum()
n_dupe_ids = listings["listing_id"].duplicated().sum()
print(f"Uniqueness: {n_dupe_rows} duplicate rows, {n_dupe_ids} duplicate listing_id values")

# Completeness: Check for missing values in each column
missing = listings.isnull().sum()
print("\nCompleteness (missing values):")
print(missing[missing > 0].sort_values(ascending=False))

# Consistency: Check for categorical columns with a fixed vocabulary, and report any unexpected values
for col in ["room_type", "host_is_superhost", "instant_bookable"]:
    print(f"\nConsistency — {col}: {listings[col].unique()}")

# Accuracy: Check for values that are structurally impossible, not just statistically unusual
bad_price = (listings["price"] <= 0).sum()
bad_accommodates = (listings["accommodates"] < 1).sum()
bad_coords = (~listings["latitude"].between(-90, 90) | ~listings["longitude"].between(-180, 180)).sum()
print(f"\nAccuracy: {bad_price} price<=0, {bad_accommodates} accommodates<1, {bad_coords} out-of-range coordinates")

**Interpretation: Data Quality Check**

Checking the data for completeness, uniqueness, consistency, and accuracy surfaces several real issues that shape the cleaning plan in Data Preparation:

- **Uniqueness:** zero duplicate rows and zero duplicate `listing_id` values - no duplication problem to clean up.
- **Completeness:** `district` is missing on 242,700 of 279,712 rows. `host_response_time` / `host_response_rate` are each missing on 128,782 rows; `host_acceptance_rate` on 113,087 rows; each `review_scores_*` column on roughly 91,000–92,000 rows; `bedrooms` on 29,435 rows. A smaller cluster of host/name fields is missing on 165–840 rows each.
- **Consistency:** the three key categorical columns hold a clean, fixed vocabulary with no stray values - `room_type` is exactly `{Entire place, Private room, Hotel room, Shared room}`, `host_is_superhost` is `{f, t, missing}`, and `instant_bookable` is `{f, t}`.
- **Accuracy:** checking for structurally impossible values (not just statistically unusual ones) found 113 rows with `price <= 0`, 85 rows with `accommodates < 1`, and zero rows with out-of-range latitude/longitude. The 113 bad-price rows are consistent with, and slightly refine, the `price <= 0` drop rule used in Data Preparation.

### The key discovery: price is in local currency, not a shared one

This single finding shapes the entire modeling strategy. One global model trained across every city
would have mostly learned which city a record comes from rather than real pricing dynamics.

In [ ]:
price_by_city = listings.groupby("city")["price"].agg(["mean", "median", "min", "max"]).round(1)
price_by_city

**Interpretation:** every market's mean sits well above its median (e.g. Bangkok mean 2,078 vs.
median 1,100; Rio de Janeiro mean 743 vs. median 280), a classic right-skew signature where a small
number of very expensive listings pull the average up. Minimums of 0 in several markets are data
errors or inactive listings rather than real free stays, and will be dropped in Data Preparation.
Maximums range from roughly 10,000 (Paris, Rome) to nearly 500,000 (Mexico City), confirming that
price scale is market-specific rather than just a currency conversion factor, reinforcing the
per-market modeling decision above.

In [ ]:
# Check category balance for room_type, and quantify missing data across all columns
print(listings["room_type"].value_counts())
print()

**Room type balance:** "Entire place" dominates at 182,005 rows, "Private room" is the
next largest at 86,988, and "Hotel room" / "Shared room" are small minority classes
(5,857 and 4,862 respectively).<br>

In [ ]:
# Zoom into a single market (Bangkok) to quantify the outlier severity
bkk = listings[listings["city"] == "Bangkok"]["price"]
print(bkk.describe())
print(f"\n99th percentile: {bkk.quantile(0.99):,.0f}   |   max: {bkk.max():,.0f}")

**Interpretation:** Bangkok's mean (2,078) is nearly double its median (1,100), and the 99th
percentile (15,553) is more than 19 times smaller than the actual maximum (300,177). A handful of
extreme listings are dragging the mean, and would dominate a model trained on raw price.